# Cross-Modal Face Recognition: Sketch-to-Photo
### Comparative Analysis — resnet ResNet50 vs LightCNN-29

This notebook trains and evaluates two models on the task of matching face sketches to photographs.
Both models are trained with the same pipeline: CUFS pretraining → FS2K fine-tuning using batch-hard triplet loss.

## 1. Configuration

In [1]:
FS2K_DIR = "/datasets/FS2K/"
CUFS_SKETCH_DIR = "/datasets/CUFS/sketches/"
CUFS_PHOTO_DIR  = "/datasets/CUFS/photos/"
SAVE_RESNET    = "/resnet_checkpoint.pth"
SAVE_VGG  = "/vgg_checkpoint.pth"

## 2. Imports

In [2]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)
print('Torch:', torch.__version__)

Using: cpu
Torch: 2.0.1+cpu


## 3. Data Loading

**CUFS** uses a filename-based mapping (e.g. `f-039-sz1.jpg` → `f-039.jpg`).  
**FS2K** uses a folder structure with `photo1/2/3` and `sketch1/2/3` subfolders.

In [3]:
def map_sketch_to_photo(sketch_id):
    sketch_id = sketch_id.replace('.jpg', '').replace('-sz1', '')
    if sketch_id.startswith('F2-'):  return sketch_id.replace('F2-', 'f-')
    if sketch_id.startswith('f1-') or sketch_id.startswith('f-'): return sketch_id
    if sketch_id.startswith('M2-'):  return sketch_id.replace('M2-', 'm-')
    if sketch_id.startswith('m1-') or sketch_id.startswith('m-'): return sketch_id
    return None


class CUFSDataset(Dataset):
    def __init__(self, sketch_dir, photo_dir, transform=None):
        self.sketch_dir = sketch_dir
        self.photo_dir  = photo_dir
        self.transform  = transform
        photo_map = {p.replace('.jpg', ''): p for p in os.listdir(photo_dir)}
        self.pairs = []
        for s in os.listdir(sketch_dir):
            pid = map_sketch_to_photo(s)
            if pid and pid in photo_map:
                self.pairs.append((s, photo_map[pid]))
        print(f'CUFS: {len(self.pairs)} pairs found')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        sn, pn = self.pairs[idx]
        sketch = Image.open(os.path.join(self.sketch_dir, sn)).convert('RGB')
        photo  = Image.open(os.path.join(self.photo_dir,  pn)).convert('RGB')
        if self.transform:
            sketch = self.transform(sketch)
            photo  = self.transform(photo)
        return sketch, photo, idx


class FS2KDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.pairs = []
        for split in ['1', '2', '3']:
            pd = os.path.join(root_dir, 'photo',  f'photo{split}')
            sd = os.path.join(root_dir, 'sketch', f'sketch{split}')
            if not os.path.exists(pd) or not os.path.exists(sd):
                continue
            for fname in sorted(os.listdir(pd)):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue
                sp = os.path.join(sd, fname.replace('image', 'sketch'))
                if os.path.exists(sp):
                    self.pairs.append((sp, os.path.join(pd, fname)))
        print(f'FS2K: {len(self.pairs)} pairs found')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        sp, pp = self.pairs[idx]
        sketch = Image.open(sp).convert('RGB')
        photo  = Image.open(pp).convert('RGB')
        if self.transform:
            sketch = self.transform(sketch)
            photo  = self.transform(photo)
        return sketch, photo, idx

## 4. Preprocessing

Images are resized to 112×112 and normalized to [-1, 1]. Training sketches receive augmentation (flip, brightness, rotation) to reduce overfitting on the small dataset.

In [4]:
def pil_to_tensor(img):
    """Convert PIL image to float tensor without numpy/torchvision."""
    img = img.resize((112, 112), Image.BILINEAR).convert('RGB')
    pixels = list(img.getdata())
    r = torch.tensor([p[0] for p in pixels], dtype=torch.float32).reshape(112, 112)
    g = torch.tensor([p[1] for p in pixels], dtype=torch.float32).reshape(112, 112)
    b = torch.tensor([p[2] for p in pixels], dtype=torch.float32).reshape(112, 112)
    return (torch.stack([r, g, b]) / 127.5) - 1.0

def transform_train(img):
    img = img.resize((112, 112), Image.BILINEAR).convert('RGB')
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() > 0.5:
        f = random.uniform(0.7, 1.3)
        img = img.point(lambda p: min(255, int(p * f)))
    if random.random() > 0.5:
        img = img.rotate(random.uniform(-15, 15))
    pixels = list(img.getdata())
    r = torch.tensor([p[0] for p in pixels], dtype=torch.float32).reshape(112, 112)
    g = torch.tensor([p[1] for p in pixels], dtype=torch.float32).reshape(112, 112)
    b = torch.tensor([p[2] for p in pixels], dtype=torch.float32).reshape(112, 112)
    return (torch.stack([r, g, b]) / 127.5) - 1.0

def transform_test(img):
    return pil_to_tensor(img)

# Build datasets
cufs_ds      = CUFSDataset(CUFS_SKETCH_DIR, CUFS_PHOTO_DIR, transform_train)
print(f'Total training pairs: CUFS={len(cufs_ds)}')

random.seed(42)  # reproducible split

full_train = FS2KDataset(FS2K_DIR, transform_train)
full_test  = FS2KDataset(FS2K_DIR, transform_test)

N       = len(full_train)
indices = list(range(N))
random.shuffle(indices)

split      = int(0.8 * N)
train_idx  = indices[:split]
test_idx   = indices[split:]

from torch.utils.data import Subset
fs2k_ds_train = Subset(full_train, train_idx)
fs2k_ds_test  = Subset(full_test,  test_idx)

print(f'FS2K Train: {len(fs2k_ds_train)} | Test: {len(fs2k_ds_test)}')

CUFS: 188 pairs found
Total training pairs: CUFS=188
FS2K: 2006 pairs found
FS2K: 2006 pairs found
FS2K Train: 1604 | Test: 402


## 5. Loss Function — Batch-Hard Triplet Loss

For each sketch (anchor), the hardest negative (most similar wrong photo) is selected within the batch. This ensures the model always trains on informative examples.

In [5]:
def batch_hard_triplet_loss(sketch_embs, photo_embs, margin=0.3):
    B   = sketch_embs.size(0)
    sim = torch.mm(sketch_embs, photo_embs.t())
    total_loss = torch.tensor(0.0, device=sketch_embs.device, requires_grad=True)
    count = 0
    for i in range(B):
        pos_sim  = sim[i, i]
        neg_sims = torch.cat([sim[i, :i], sim[i, i+1:]])
        hard_neg = neg_sims.max()
        triplet  = margin - pos_sim + hard_neg
        if triplet.item() > 0:
            total_loss = total_loss + triplet
            count += 1
    if count == 0:
        return torch.tensor(0.0, device=sketch_embs.device, requires_grad=True)
    return total_loss / count


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for sketches, photos, _ in loader:
        sketches, photos = sketches.to(device), photos.to(device)
        s_emb = F.normalize(model(sketches), p=2, dim=1)
        p_emb = F.normalize(model(photos),   p=2, dim=1)
        loss  = batch_hard_triplet_loss(s_emb, p_emb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

## 6. Evaluation Function

Reports four standard metrics used in heterogeneous face recognition literature:
- **Rank-1 / Rank-5 / Rank-10**: retrieval accuracy at different cutoffs
- **ROC AUC**: overall separability between genuine and impostor pairs
- **TAR@FAR=1%**: true accept rate at a 1% false accept rate operating point

In [6]:
def evaluate(model, dataset, model_name):
    model.eval()
    all_s, all_p = [], []
    with torch.no_grad():
        for sketches, photos, _ in DataLoader(dataset, batch_size=32, num_workers=0):
            all_s.append(F.normalize(model(sketches.to(device)), dim=1))
            all_p.append(F.normalize(model(photos.to(device)),   dim=1))
    all_s = torch.cat(all_s)
    all_p = torch.cat(all_p)
    N     = all_s.size(0)

    sim_matrix = torch.mm(all_s, all_p.t())
    ranks      = sim_matrix.argsort(dim=1, descending=True)
    correct    = torch.arange(N, device=device)

    rank1  = (ranks[:, 0] == correct).float().mean().item()
    rank5  = sum(correct[i].item() in ranks[i, :5].tolist()  for i in range(N)) / N
    rank10 = sum(correct[i].item() in ranks[i, :10].tolist() for i in range(N)) / N

    sims_flat   = sim_matrix.cpu().flatten().tolist()
    gt_flat     = [1 if i == j else 0 for i in range(N) for j in range(N)]
    auc         = roc_auc_score(gt_flat, sims_flat)
    fpr, tpr, _ = roc_curve(gt_flat, sims_flat)
    tar_far1    = float(tpr[next(i for i, f in enumerate(fpr) if f >= 0.01)])

    print(f'\n{"="*45}')
    print(f'  {model_name} — Results on FS2K ({N} pairs)')
    print(f'{"="*45}')
    print(f'  Rank-1  Accuracy : {rank1*100:6.2f}%')
    print(f'  Rank-5  Accuracy : {rank5*100:6.2f}%')
    print(f'  Rank-10 Accuracy : {rank10*100:6.2f}%')
    print(f'  ROC AUC          : {auc:.4f}')
    print(f'  TAR@FAR=1%       : {tar_far1*100:6.2f}%')
    print(f'{"="*45}')
    return {'model': model_name, 'rank1': rank1, 'rank5': rank5,
            'rank10': rank10, 'auc': auc, 'tar_far1': tar_far1}

---
## 7. Model A — resnet ResNet50

ResNet50 pretrained on ImageNet, with a 512-d projection head. The backbone provides strong general visual features; the head is fine-tuned to produce face embeddings.

**Training strategy:**
1. Phase 1 (CUFS) — freeze backbone, train head only at lr=1e-3
2. Phase 2 (FS2K) — unfreeze last ResNet block, train with lower lr
3. Phase 3 (FS2K extended) — unfreeze more layers, train with augmentation at very low lr

In [ ]:
import torchvision.models as tvm

class ResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = tvm.resnet50(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.head = nn.Linear(2048, 512)

    def forward(self, x):
        return self.head(self.backbone(x).flatten(1))

model = ResNet50().to(device)

# load checkpoint if it exists
if os.path.exists(SAVE_RESNET):
    model.load_state_dict(torch.load(SAVE_RESNET, map_location=device))
    print('Loaded existing resnet checkpoint')
else:
    print('resnet: starting from ImageNet weights')

print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

Loaded existing ArcFace checkpoint
Total params: 24,557,120


In [ ]:
# Phase 1: CUFS pretraining (head only) 
for p in model.backbone.parameters(): p.requires_grad = False
for p in model.head.parameters():     p.requires_grad = True

cufs_loader = DataLoader(cufs_ds, batch_size=8, shuffle=True, num_workers=0)
optimizer   = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

print('=== RESNET-50 Phase 1: CUFS pretraining ===')
for epoch in range(1, 11):  # 10 epochs
    start = time.time()
    loss  = train_one_epoch(model, cufs_loader, optimizer)
    print(f'  Epoch {epoch:02d}/10 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)

=== ArcFace Phase 1: CUFS pretraining ===
  Epoch 01/10 — Loss: 0.3102 — 0.3 min
  Epoch 02/10 — Loss: 0.3010 — 0.2 min
  Epoch 03/10 — Loss: 0.2993 — 0.2 min
  Epoch 04/10 — Loss: 0.2995 — 0.2 min
  Epoch 05/10 — Loss: 0.2990 — 0.2 min
  Epoch 06/10 — Loss: 0.2957 — 0.2 min
  Epoch 07/10 — Loss: 0.2853 — 0.2 min
  Epoch 08/10 — Loss: 0.3058 — 0.2 min
  Epoch 09/10 — Loss: 0.3009 — 0.2 min
  Epoch 10/10 — Loss: 0.2976 — 0.2 min


In [ ]:
# Phase 2: FS2K fine-tuning (unfreeze layer4 + head) 
for p in model.backbone[-2].parameters(): p.requires_grad = True

optimizer = torch.optim.Adam([
    {'params': model.backbone[-2].parameters(), 'lr': 1e-5},
    {'params': model.head.parameters(),         'lr': 1e-4},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== resnet Phase 2: FS2K fine-tuning ===')
for epoch in range(1, 21):  # 20 epochs
    start = time.time()
    loss  = train_one_epoch(model, fs2k_loader, optimizer)
    print(f'  Epoch {epoch:02d}/20 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)

=== ArcFace Phase 2: FS2K fine-tuning ===
  Epoch 01/20 — Loss: 0.3020 — 2.8 min
  Epoch 02/20 — Loss: 0.2908 — 2.7 min
  Epoch 03/20 — Loss: 0.2722 — 2.7 min
  Epoch 04/20 — Loss: 0.2537 — 2.7 min
  Epoch 05/20 — Loss: 0.2398 — 2.7 min
  Epoch 06/20 — Loss: 0.2344 — 2.7 min
  Epoch 07/20 — Loss: 0.2256 — 2.7 min
  Epoch 08/20 — Loss: 0.2172 — 2.7 min
  Epoch 09/20 — Loss: 0.2128 — 2.6 min
  Epoch 10/20 — Loss: 0.2116 — 2.9 min
  Epoch 11/20 — Loss: 0.1984 — 2.7 min
  Epoch 12/20 — Loss: 0.1958 — 2.5 min
  Epoch 13/20 — Loss: 0.1998 — 2.4 min
  Epoch 14/20 — Loss: 0.1807 — 2.6 min
  Epoch 15/20 — Loss: 0.1680 — 2.4 min
  Epoch 16/20 — Loss: 0.1777 — 2.4 min
  Epoch 17/20 — Loss: 0.1696 — 2.4 min
  Epoch 18/20 — Loss: 0.1766 — 2.5 min
  Epoch 19/20 — Loss: 0.1632 — 2.3 min
  Epoch 20/20 — Loss: 0.1696 — 2.2 min


In [ ]:
# Phase 3: Extended FS2K training (unfreeze more, very low lr) 
for p in model.backbone[-4:].parameters(): p.requires_grad = True

optimizer = torch.optim.Adam([
    {'params': model.backbone[-4:].parameters(), 'lr': 5e-6},
    {'params': model.head.parameters(),          'lr': 5e-5},
])

print('=== resnet Phase 3: Extended FS2K training ===')
for epoch in range(1, 31):  # 30 epochs
    start = time.time()
    loss  = train_one_epoch(model, fs2k_loader, optimizer)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(model.state_dict(), SAVE_RESNET)
    if loss < 0.01:  # stop early if converged
        print('  Converged — stopping early')
        break

=== ArcFace Phase 3: Extended FS2K training ===
  Epoch 01/30 — Loss: 0.1535 — 3.5 min
  Epoch 02/30 — Loss: 0.1469 — 3.4 min
  Epoch 03/30 — Loss: 0.1364 — 3.6 min
  Epoch 04/30 — Loss: 0.1434 — 3.5 min
  Epoch 05/30 — Loss: 0.1460 — 3.4 min
  Epoch 06/30 — Loss: 0.1327 — 3.4 min
  Epoch 07/30 — Loss: 0.1200 — 3.5 min
  Epoch 08/30 — Loss: 0.1338 — 3.7 min
  Epoch 09/30 — Loss: 0.1095 — 3.6 min
  Epoch 10/30 — Loss: 0.1010 — 3.5 min
  Epoch 11/30 — Loss: 0.0991 — 3.3 min
  Epoch 12/30 — Loss: 0.1142 — 3.3 min
  Epoch 13/30 — Loss: 0.1020 — 3.2 min
  Epoch 14/30 — Loss: 0.1177 — 3.3 min
  Epoch 15/30 — Loss: 0.1032 — 3.1 min
  Epoch 16/30 — Loss: 0.0778 — 3.1 min
  Epoch 17/30 — Loss: 0.0959 — 3.2 min
  Epoch 18/30 — Loss: 0.0878 — 3.1 min
  Epoch 19/30 — Loss: 0.0971 — 3.8 min
  Epoch 20/30 — Loss: 0.0908 — 6.0 min
  Epoch 21/30 — Loss: 0.0775 — 5.9 min
  Epoch 22/30 — Loss: 0.0764 — 6.4 min
  Epoch 23/30 — Loss: 0.0849 — 6.6 min
  Epoch 24/30 — Loss: 0.0782 — 6.6 min
  Epoch 25/30 — 

In [ ]:
# Evaluate resnet
model.load_state_dict(torch.load(SAVE_RESNET, map_location=device))
results_resnet = evaluate(model, fs2k_ds_test, 'resnet ResNet50')


  ArcFace ResNet50 — Results on FS2K (402 pairs)
  Rank-1  Accuracy :  27.86%
  Rank-5  Accuracy :  57.21%
  Rank-10 Accuracy :  70.40%
  ROC AUC          : 0.9628
  TAR@FAR=1%       :  43.28%


---
## 8. Model B — VGG-16

VGG-16 uses Max Feature Map (MFM) activations instead of ReLU, which suppress noisy activations and select the most discriminative features. It was specifically designed for heterogeneous face recognition and is a standard baseline in sketch-photo literature.


In [9]:
#VGG16 Baseline 
class VGGBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tvm.vgg16(weights='IMAGENET1K_V1')
        self.backbone = vgg.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Linear(512, 512)

    def forward(self, x):
        x = self.pool(self.backbone(x)).flatten(1)
        return self.head(x)

vgg_model = VGGBaseline().to(device)

if os.path.exists(SAVE_VGG):
    vgg_model.load_state_dict(torch.load(SAVE_VGG, map_location=device))
    print('Loaded existing VGG checkpoint')
else:
    print('VGG16: starting from ImageNet weights')

# freeze backbone, train head only for phase 1
for p in vgg_model.backbone.parameters(): p.requires_grad = False
for p in vgg_model.head.parameters():     p.requires_grad = True

print(f'Trainable params: {sum(p.numel() for p in vgg_model.parameters() if p.requires_grad):,}')

Loaded existing VGG checkpoint
Trainable params: 262,656


In [30]:
cufs_loader = DataLoader(cufs_ds, batch_size=8, shuffle=True, num_workers=0)
optimizer_v = torch.optim.Adam(filter(lambda p: p.requires_grad, vgg_model.parameters()), lr=1e-3)

print('=== VGG16 Phase 1: CUFS pretraining ===')
for epoch in range(1, 11):
    start = time.time()
    loss  = train_one_epoch(vgg_model, cufs_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/10 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)

=== VGG16 Phase 1: CUFS pretraining ===
  Epoch 01/10 — Loss: 0.3263 — 0.6 min
  Epoch 02/10 — Loss: 0.3044 — 0.5 min
  Epoch 03/10 — Loss: 0.2992 — 0.5 min
  Epoch 04/10 — Loss: 0.2975 — 0.5 min
  Epoch 05/10 — Loss: 0.2938 — 0.5 min
  Epoch 06/10 — Loss: 0.2931 — 0.5 min
  Epoch 07/10 — Loss: 0.2976 — 0.5 min
  Epoch 08/10 — Loss: 0.2914 — 0.5 min
  Epoch 09/10 — Loss: 0.2930 — 0.5 min
  Epoch 10/10 — Loss: 0.2888 — 0.5 min


In [31]:
# unfreeze last few VGG conv layers
for p in vgg_model.backbone[-6:].parameters(): p.requires_grad = True

optimizer_v = torch.optim.Adam([
    {'params': vgg_model.backbone[-6:].parameters(), 'lr': 1e-5},
    {'params': vgg_model.head.parameters(),          'lr': 1e-4},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== VGG16 Phase 2: FS2K fine-tuning ===')
for epoch in range(1, 21):
    start = time.time()
    loss  = train_one_epoch(vgg_model, fs2k_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)
    if loss < 0.01:
        print('  Converged — stopping early')
        break

=== VGG16 Phase 2: FS2K fine-tuning ===
  Epoch 01/30 — Loss: 0.3080 — 4.7 min
  Epoch 02/30 — Loss: 0.2870 — 4.6 min
  Epoch 03/30 — Loss: 0.2717 — 4.6 min
  Epoch 04/30 — Loss: 0.2587 — 4.6 min
  Epoch 05/30 — Loss: 0.2453 — 4.8 min
  Epoch 06/30 — Loss: 0.2373 — 4.7 min
  Epoch 07/30 — Loss: 0.2284 — 4.5 min
  Epoch 08/30 — Loss: 0.2270 — 4.5 min
  Epoch 09/30 — Loss: 0.2190 — 4.4 min
  Epoch 10/30 — Loss: 0.2087 — 4.4 min
  Epoch 11/30 — Loss: 0.2013 — 4.4 min
  Epoch 12/30 — Loss: 0.1984 — 4.5 min
  Epoch 13/30 — Loss: 0.1904 — 4.5 min
  Epoch 14/30 — Loss: 0.1811 — 4.4 min
  Epoch 15/30 — Loss: 0.1888 — 4.4 min
  Epoch 16/30 — Loss: 0.1885 — 4.5 min
  Epoch 17/30 — Loss: 0.1794 — 4.4 min
  Epoch 18/30 — Loss: 0.1713 — 4.4 min
  Epoch 19/30 — Loss: 0.1684 — 4.5 min
  Epoch 20/30 — Loss: 0.1739 — 4.5 min


In [32]:
# unfreeze more VGG layers for phase 3
for p in vgg_model.backbone[-10:].parameters():
    p.requires_grad = True

optimizer_v = torch.optim.Adam([
    {'params': vgg_model.backbone[-10:].parameters(), 'lr': 5e-6},
    {'params': vgg_model.head.parameters(),           'lr': 5e-5},
])

fs2k_loader = DataLoader(fs2k_ds_train, batch_size=8, shuffle=True, num_workers=0)

print('=== VGG16 Phase 3: Extended FS2K training ===')
for epoch in range(1, 31):
    start = time.time()
    loss  = train_one_epoch(vgg_model, fs2k_loader, optimizer_v)
    print(f'  Epoch {epoch:02d}/30 — Loss: {loss:.4f} — {(time.time()-start)/60:.1f} min')
    torch.save(vgg_model.state_dict(), SAVE_VGG)
    if loss < 0.01:
        print('  Converged — stopping early')
        break

=== VGG16 Phase 3: Extended FS2K training ===
  Epoch 01/30 — Loss: 0.1649 — 5.3 min
  Epoch 02/30 — Loss: 0.1660 — 5.2 min
  Epoch 03/30 — Loss: 0.1593 — 5.2 min
  Epoch 04/30 — Loss: 0.1604 — 5.2 min
  Epoch 05/30 — Loss: 0.1578 — 5.2 min
  Epoch 06/30 — Loss: 0.1530 — 5.2 min
  Epoch 07/30 — Loss: 0.1519 — 5.2 min
  Epoch 08/30 — Loss: 0.1505 — 5.2 min
  Epoch 09/30 — Loss: 0.1505 — 5.2 min
  Epoch 10/30 — Loss: 0.1416 — 5.2 min
  Epoch 11/30 — Loss: 0.1454 — 5.2 min
  Epoch 12/30 — Loss: 0.1436 — 5.2 min
  Epoch 13/30 — Loss: 0.1487 — 5.3 min
  Epoch 14/30 — Loss: 0.1390 — 5.2 min
  Epoch 15/30 — Loss: 0.1344 — 5.2 min
  Epoch 16/30 — Loss: 0.1317 — 5.2 min
  Epoch 17/30 — Loss: 0.1380 — 5.2 min
  Epoch 18/30 — Loss: 0.1279 — 5.2 min
  Epoch 19/30 — Loss: 0.1396 — 5.1 min
  Epoch 20/30 — Loss: 0.1299 — 5.2 min
  Epoch 21/30 — Loss: 0.1273 — 5.2 min
  Epoch 22/30 — Loss: 0.1257 — 5.2 min
  Epoch 23/30 — Loss: 0.1214 — 4.2 min
  Epoch 24/30 — Loss: 0.1285 — 4.1 min
  Epoch 25/30 — Lo

In [10]:
vgg_model.load_state_dict(torch.load(SAVE_VGG, map_location=device))
results_vgg = evaluate(vgg_model, fs2k_ds_test, 'VGG16 Baseline')


  VGG16 Baseline — Results on FS2K (402 pairs)
  Rank-1  Accuracy :  53.73%
  Rank-5  Accuracy :  81.59%
  Rank-10 Accuracy :  89.30%
  ROC AUC          : 0.9866
  TAR@FAR=1%       :  73.13%


---
## 9. Comparison Table

In [ ]:
print('\n' + '='*55)
print(f'{"Metric":<20} {"resnet ResNet50":>16} {"VGG Baseline":>14}')
print('='*55)
for key, label in [
    ('rank1',    'Rank-1'),
    ('rank5',    'Rank-5'),
    ('rank10',   'Rank-10'),
    ('auc',      'ROC AUC'),
    ('tar_far1', 'TAR@FAR=1%'),
]:
    a = results_resnet[key] * 100
    l = results_vgg[key] * 100
    winner = '<' if a > l else '>'
    print(f'{label:<20} {a:>15.2f}% {l:>13.2f}%  {winner}')
print('='*55)
print('< = ResNet-50 wins   > = VGG wins')


Metric               ArcFace ResNet50   VGG Baseline
Rank-1                         27.86%         53.73%  >
Rank-5                         57.21%         81.59%  >
Rank-10                        70.40%         89.30%  >
ROC AUC                        96.28%         98.66%  >
TAR@FAR=1%                     43.28%         73.13%  >
< = ArcFace wins   > = VGG wins


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os

def get_embeddings(mdl, dataset):
    mdl.eval()
    all_s, all_p = [], []
    with torch.no_grad():
        for sketches, photos, _ in DataLoader(dataset, batch_size=32, num_workers=0):
            all_s.append(F.normalize(mdl(sketches.to(device)), dim=1))
            all_p.append(F.normalize(mdl(photos.to(device)),   dim=1))
    return torch.cat(all_s), torch.cat(all_p)

# load best weights
model.load_state_dict(torch.load(SAVE_RESNET,  map_location=device))
vgg_model.load_state_dict(torch.load(SAVE_VGG,  map_location=device))

arc_s, arc_p = get_embeddings(model,     fs2k_ds_test)
vgg_s, vgg_p = get_embeddings(vgg_model, fs2k_ds_test)
N = arc_s.size(0)

arc_sim = torch.mm(arc_s, arc_p.t())
vgg_sim = torch.mm(vgg_s, vgg_p.t())

gt_flat  = [1 if i == j else 0 for i in range(N) for j in range(N)]
arc_flat = arc_sim.cpu().flatten().tolist()
vgg_flat = vgg_sim.cpu().flatten().tolist()

# ROC Curve
arc_fpr, arc_tpr, _ = roc_curve(gt_flat, arc_flat)
vgg_fpr, vgg_tpr, _ = roc_curve(gt_flat, vgg_flat)
arc_auc = roc_auc_score(gt_flat, arc_flat)
vgg_auc = roc_auc_score(gt_flat, vgg_flat)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(arc_fpr, arc_tpr, color='#E24B4A', linewidth=2,
        label=f'ArcFace ResNet-50 (AUC = {arc_auc:.4f})')
ax.plot(vgg_fpr, vgg_tpr, color='#1D9E75', linewidth=2,
        label=f'VGG-16 (AUC = {vgg_auc:.4f})')
ax.plot([0,1],[0,1],'--', color='#888780', linewidth=1, label='Random chance')
ax.axvline(x=0.01, color='#BA7517', linewidth=1, linestyle=':', label='FAR = 1%')
ax.set_xlabel('False Accept Rate (FAR)', fontsize=12)
ax.set_ylabel('True Accept Rate (TAR)', fontsize=12)
ax.set_title('ROC Curve — Sketch-to-Photo Verification', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig/roc_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: roc_curve.png')

#CMC Curve
ranks_to_plot = list(range(1, 21))

def cmc(sim_matrix, max_rank=20):
    ranks = sim_matrix.argsort(dim=1, descending=True)
    correct = torch.arange(N, device=device)
    rates = []
    for k in range(1, max_rank+1):
        rate = sum(correct[i].item() in ranks[i, :k].tolist()
                   for i in range(N)) / N
        rates.append(rate * 100)
    return rates

arc_cmc = cmc(arc_sim)
vgg_cmc = cmc(vgg_sim)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ranks_to_plot, arc_cmc, color='#E24B4A', marker='o',
        markersize=4, linewidth=2, label='ArcFace ResNet-50')
ax.plot(ranks_to_plot, vgg_cmc, color='#1D9E75', marker='s',
        markersize=4, linewidth=2, label='VGG-16')
ax.set_xlabel('Rank', fontsize=12)
ax.set_ylabel('Cumulative Match Rate (%)', fontsize=12)
ax.set_title('CMC Curve — Sketch-to-Photo Retrieval', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim([1, 20]); ax.set_ylim([0, 100])
ax.set_xticks(ranks_to_plot)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig/cmc_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: cmc_curve.png')

# Score Distribution (Genuine vs Impostor)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sim_flat, name, color in [
    (axes[0], arc_flat, 'ArcFace ResNet-50', '#E24B4A'),
    (axes[1], vgg_flat, 'VGG-16',            '#1D9E75'),
]:
    genuine   = [s for s, g in zip(sim_flat, gt_flat) if g == 1]
    impostor  = [s for s, g in zip(sim_flat, gt_flat) if g == 0]
    ax.hist(impostor, bins=80, alpha=0.6, color='#888780',
            label='Impostor pairs', density=True)
    ax.hist(genuine,  bins=80, alpha=0.7, color=color,
            label='Genuine pairs',  density=True)
    ax.set_xlabel('Cosine Similarity', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'Score Distribution — {name}', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig/score_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: score_distribution.png')

# Bar Chart
arc_results = {'Rank-1': 27.86, 'Rank-5': 57.21,
               'Rank-10': 70.40, 'AUC×100': 96.28, 'TAR@FAR=1%': 43.28}
vgg_results = {'Rank-1': 53.82,   'Rank-5': 81.59,
               'Rank-10': 89.30,  'AUC×100': 98.66,   'TAR@FAR=1%': 73.13}

labels  = list(arc_results.keys())
arc_vals = list(arc_results.values())
vgg_vals = list(vgg_results.values())

x = range(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar([i - w/2 for i in x], arc_vals, w,
               color='#E24B4A', alpha=0.85, label='ArcFace ResNet-50')
bars2 = ax.bar([i + w/2 for i in x], vgg_vals, w,
               color='#1D9E75', alpha=0.85, label='VGG-16')

for bar in bars1 + bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.8,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Performance Comparison — ArcFace ResNet-50 vs VGG-16', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim([0, 110])
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('fig/metric_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: metric_comparison.png')

# Loss Curves 
arc_losses_p1 = [
    0.3102, 0.3010, 0.2993, 0.2995, 0.2990,
    0.2957, 0.2853, 0.3058, 0.3009, 0.2976
]

arc_losses_p2 = [
    0.3020, 0.2908, 0.2722, 0.2537, 0.2398,
    0.2344, 0.2256, 0.2172, 0.2128, 0.2116,
    0.1984, 0.1958, 0.1998, 0.1807, 0.1680,
    0.1777, 0.1696, 0.1766, 0.1632, 0.1696
]

arc_losses_p3 = [
    0.1535, 0.1469, 0.1364, 0.1434, 0.1460,
    0.1327, 0.1200, 0.1338, 0.1095, 0.1010,
    0.0991, 0.1142, 0.1020, 0.1177, 0.1032,
    0.0778, 0.0959, 0.0878, 0.0971, 0.0908,
    0.0775, 0.0764, 0.0849, 0.0782, 0.0621,
    0.0721, 0.0735, 0.0655, 0.0612, 0.0658
]

vgg_losses_p1 = [
    0.3263, 0.3044, 0.2992, 0.2975, 0.2938,
    0.2931, 0.2976, 0.2914, 0.2930, 0.2888
]

vgg_losses_p2 = [
    0.3080, 0.2870, 0.2717, 0.2587, 0.2453,
    0.2373, 0.2284, 0.2270, 0.2190, 0.2087,
    0.2013, 0.1984, 0.1904, 0.1811, 0.1888,
    0.1885, 0.1794, 0.1713, 0.1684, 0.1739
]

vgg_losses_p3 = [
    0.1649, 0.1660, 0.1593, 0.1604, 0.1578,
    0.1530, 0.1519, 0.1505, 0.1505, 0.1416,
    0.1454, 0.1436, 0.1487, 0.1390, 0.1344,
    0.1317, 0.1380, 0.1279, 0.1396, 0.1299,
    0.1273, 0.1257, 0.1214, 0.1285, 0.1408,
    0.1215, 0.1191, 0.1334, 0.1199, 0.1203
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, p1, p2, p3, name, color in [
    (axes[0], arc_losses_p1, arc_losses_p2, arc_losses_p3,
     'ArcFace ResNet-50', '#E24B4A'),
    (axes[1], vgg_losses_p1, vgg_losses_p2, vgg_losses_p3,
     'VGG-16', '#1D9E75'),
]:
    all_losses = p1 + p2 + p3
    if not all_losses:
        ax.set_title(f'Loss Curve — {name}\n(no data yet)', fontsize=12)
        continue
    epochs = list(range(1, len(all_losses)+1))
    ax.plot(epochs, all_losses, color=color, linewidth=1.8)

    # phase boundary lines
    if p1:
        ax.axvline(x=len(p1)+0.5, color='#888780',
                   linewidth=1, linestyle='--', alpha=0.6)
    if p1 and p2:
        ax.axvline(x=len(p1)+len(p2)+0.5, color='#888780',
                   linewidth=1, linestyle='--', alpha=0.6)

    # phase labels
    def mid(start, length):
        return start + length/2

    if p1:
        ax.text(mid(0, len(p1)), max(all_losses)*0.95,
                'Phase 1\nCUFS', ha='center', fontsize=8, color='#5F5E5A')
    if p2:
        ax.text(mid(len(p1), len(p2)), max(all_losses)*0.95,
                'Phase 2\nFS2K', ha='center', fontsize=8, color='#5F5E5A')
    if p3:
        ax.text(mid(len(p1)+len(p2), len(p3)), max(all_losses)*0.95,
                'Phase 3\nFS2K ext.', ha='center', fontsize=8, color='#5F5E5A')

    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Triplet Loss', fontsize=11)
    ax.set_title(f'Loss Curve — {name}', fontsize=12)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig/loss_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: loss_curves.png')

print('\nAll graphs saved:')
print('  roc_curve.png')
print('  cmc_curve.png')
print('  score_distribution.png')
print('  metric_comparison.png')
print('  loss_curves.png')

C:\Users\Sufia\AppData\Local\Temp\ipykernel_21484\500727046.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: roc_curve.png


C:\Users\Sufia\AppData\Local\Temp\ipykernel_21484\500727046.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: cmc_curve.png


C:\Users\Sufia\AppData\Local\Temp\ipykernel_21484\500727046.py:115: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: score_distribution.png


C:\Users\Sufia\AppData\Local\Temp\ipykernel_21484\500727046.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: metric_comparison.png
Saved: loss_curves.png

All graphs saved:
  roc_curve.png
  cmc_curve.png
  score_distribution.png
  metric_comparison.png
  loss_curves.png


C:\Users\Sufia\AppData\Local\Temp\ipykernel_21484\500727046.py:244: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
def query_sketch(sketch_path, model, dataset, top_k=5):
    model.eval()

    # load query
    query       = Image.open(sketch_path).convert('RGB')
    query_tensor = transform_test(query).unsqueeze(0).to(device)

    # get query embedding
    with torch.no_grad():
        query_emb = F.normalize(model(query_tensor), p=2, dim=1)

    # build photo gallery
    all_p = []
    with torch.no_grad():
        for _, photos, _ in DataLoader(dataset, batch_size=32, num_workers=0):
            all_p.append(F.normalize(model(photos.to(device)), dim=1))
    all_p = torch.cat(all_p)

    # rank
    sims        = torch.mm(query_emb, all_p.t()).squeeze(0)
    top_indices = sims.argsort(descending=True)[:top_k]

    # display
    fig, axes = plt.subplots(1, top_k + 1, figsize=((top_k + 1) * 3, 4))

    axes[0].imshow(query.resize((112, 112)))
    axes[0].set_title('Query Sketch', fontsize=9)
    axes[0].axis('off')

    for rank, idx in enumerate(top_indices):
        idx        = idx.item()
        photo_path = dataset.dataset.pairs[dataset.indices[idx]][1]
        score      = sims[idx].item()
        photo      = Image.open(photo_path).convert('RGB').resize((112, 112))
        axes[rank + 1].imshow(photo)
        axes[rank + 1].set_title(f'Rank {rank+1}\n{score:.3f}', fontsize=9)
        axes[rank + 1].axis('off')

    plt.suptitle(f'Top {top_k} matches — {os.path.basename(sketch_path)}', fontsize=11)
    plt.tight_layout()
    plt.savefig('query_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to query_result.png')


In [20]:
index = 8
orig_idx = fs2k_ds_test.indices[index]
sketch_path = "sketch0014.png"
print(f"Using: {sketch_path}")

#query_sketch(sketch_path, model, fs2k_ds_test, top_k=10)
query_sketch(sketch_path, vgg_model, fs2k_ds_test, top_k=10)

Using: sketch0014.png
Saved to query_result.png


C:\Users\Sufia\AppData\Local\Temp\ipykernel_22172\741814984.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
